# PubMed Oncologist — DPO Training

**Phase 2:** Direct Preference Optimization fine-tuning on top of the SFT LoRA.

DPO teaches the model WHAT NOT TO SAY by contrasting good and bad responses:
- Don't fabricate specific statistics or trial results
- Use general medical knowledge freely, but flag uncertainty when going beyond well-established evidence
- Self-correct when challenged
- Maintain clinical reasoning depth

**Philosophy:** The model should behave like a well-read oncologist — drawing on broad
medical training (general knowledge) while being transparent about the strength of evidence.
It should NOT behave like a closed-book QA system that refuses to engage unless a specific
source is cited. General, well-established medical knowledge is always fair game.

**Prerequisites:**
1. Run `pubmed_datagen_v2_jupyterlab.ipynb` fully (produces SFT + DPO data)
2. Train the SFT LoRA first (Phase 1)
3. GPU with 16+ GB VRAM (DGX Spark recommended)

**Pipeline:** Base Qwen3.6-27B-FP8 → SFT LoRA (trained) → DPO LoRA (this notebook)

## 1. Setup — Configuration, Environment, GPU Check

Everything needed before training. Installs missing packages, verifies GPU, sets all paths and hyperparameters. Safe to re-run.

In [ ]:
import os, sys, subprocess, importlib, importlib.util

# DGX Spark unified memory: force CUDA allocator reclamation instead of letting
# PyTorch treat all 128GB unified memory as endlessly cacheable CUDA memory.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "garbage_collection_threshold:0.5,max_split_size_mb:256"

# ╔══════════════════════════════════════════════════════════════════════════════╗

# ║  STEP 1: INSTALL MISSING PACKAGES (safe to re-run, never clobbers NGC torch)║

# ║                                                                              ║

# ║  ORDER MATTERS:                                                              ║

# ║    1. torch check (no side effects)                                          ║

# ║    2. pip install small packages                                             ║

# ║    3. fix causal_conv1d (NGC ships broken build w/o CUDA extension)          ║

# ║    4. import unsloth FIRST (BEFORE transformers — required for patching)     ║

# ╚══════════════════════════════════════════════════════════════════════════════╝

def _pip(*args, env_extra=None):

    """Run pip with given args, suppressing output unless it fails."""

    cmd = [sys.executable, "-m", "pip"] + list(args)

    env = os.environ.copy()

    if env_extra:

        env.update(env_extra)

    result = subprocess.run(cmd, capture_output=True, text=True, env=env)

    if result.returncode != 0:

        print(f"  PIP FAILED: {' '.join(args)}")

        print(result.stderr[-500:] if result.stderr else result.stdout[-500:])

        return False

    return True

def _check_import(module_name):

    try:

        return importlib.import_module(module_name)

    except (ImportError, ModuleNotFoundError):

        return None

print("=" * 60)

print("ENVIRONMENT SETUP")

print("=" * 60)

# ── 1a. Verify NGC CUDA PyTorch is intact ──────────────────────────────────────

import torch

if not torch.cuda.is_available():

    print("FATAL: torch.cuda.is_available() = False")

    print(f"  torch version: {torch.__version__}")

    if "+cpu" in torch.__version__ or "cpu" in torch.__version__:

        print("  NGC CUDA PyTorch was clobbered by pip. Recreate the container.")

    else:

        print("  GPU not passed through. Check Portainer: runtime=nvidia, NVIDIA_VISIBLE_DEVICES=all")

    raise RuntimeError("No GPU. Cannot continue. See messages above.")

_MEMORY_FRACTION = 0.55
try:
    torch.cuda.set_per_process_memory_fraction(_MEMORY_FRACTION, 0)
    _total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"CUDA memory cap: {_total_gb * _MEMORY_FRACTION:.1f} GB ({_MEMORY_FRACTION:.0%} of {_total_gb:.0f} GB) - allocator forced to reclaim")
except RuntimeError as exc:
    print(f"WARNING: could not set CUDA memory fraction: {exc}")
    print("Relying on PYTORCH_CUDA_ALLOC_CONF garbage collection only")
print(f"  torch {torch.__version__} — CUDA {torch.version.cuda} — GPU: {torch.cuda.get_device_name(0)}")

# ── 1b. Small utility packages ─────────────────────────────────────────────────

for module, install_args in {

    "psutil":      ["install", "-q", "psutil"],

    "matplotlib":  ["install", "-q", "matplotlib"],

    "ipywidgets":  ["install", "-q", "ipywidgets"],

    "torchvision": ["install", "-q", "--no-deps", "torchvision"],

    "PIL":         ["install", "-q", "pillow"],

}.items():

    if _check_import(module) is None:

        print(f"  Installing {install_args[-1]}...")

        _pip(*install_args)

# ── 1c. Fix causal_conv1d ──────────────────────────────────────────────────────

#   NGC ships causal_conv1d 1.6.0 Python pkg WITHOUT the compiled CUDA extension

#   (causal_conv1d_cuda). This causes a hard crash when transformers or unsloth

#   try to import FalconH1 model. We must fix this BEFORE importing either.

#

#   CRITICAL: pip caches a broken prebuilt aarch64 wheel. Must use --no-binary

#   to force a source build, plus CAUSAL_CONV1D_FORCE_BUILD=TRUE env var.

#   First build takes ~3 min on aarch64, then cached by pip for future runs.

_causal_ok = False

_build_env = {

    "CAUSAL_CONV1D_FORCE_BUILD": "TRUE",

    "TORCH_CUDA_ARCH_LIST": "12.0;12.1",

}

try:

    from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn

    _causal_ok = True

    print("  causal_conv1d: OK (CUDA extension loaded)")

    for _k in list(sys.modules.keys()):

        if "causal_conv1d" in _k:

            del sys.modules[_k]

except (ImportError, ModuleNotFoundError, OSError):

    print("  causal_conv1d: CUDA extension missing — rebuilding from source (~3 min)...")

    _pip("uninstall", "-y", "causal-conv1d")

    _pip("cache", "remove", "causal_conv1d")

    for _k in list(sys.modules.keys()):

        if "causal_conv1d" in _k:

            del sys.modules[_k]

    importlib.invalidate_caches()

    ok = _pip("install", "--no-build-isolation", "--no-deps", "--force-reinstall",

              "--no-binary", "causal-conv1d", "causal-conv1d", env_extra=_build_env)

    if ok:

        importlib.invalidate_caches()

        try:

            from causal_conv1d.causal_conv1d_interface import causal_conv1d_fn

            _causal_ok = True

            print("  causal_conv1d: rebuilt OK (CUDA extension working)")

            for _k in list(sys.modules.keys()):

                if "causal_conv1d" in _k:

                    del sys.modules[_k]

        except (ImportError, ModuleNotFoundError, OSError):

            print("  causal_conv1d: rebuild produced no CUDA ext — uninstalling for fallback")

            _pip("uninstall", "-y", "causal-conv1d")

            for _k in list(sys.modules.keys()):

                if "causal_conv1d" in _k:

                    del sys.modules[_k]

            importlib.invalidate_caches()

    else:

        print("  causal_conv1d: source build failed — uninstalling for fallback")

        _pip("uninstall", "-y", "causal-conv1d")

        importlib.invalidate_caches()

# ── 1d. Import unsloth FIRST, then transformers ────────────────────────────────

#   Unsloth MUST be imported before transformers/trl/peft to apply its

#   monkey-patches. Purge any pre-loaded transformers modules.

for _k in list(sys.modules.keys()):

    if _k in ("transformers", "trl", "peft") or _k.startswith(("transformers.", "trl.", "peft.")):

        del sys.modules[_k]

importlib.invalidate_caches()

import unsloth

import transformers

print(f"  transformers {transformers.__version__}")

# ── 1e. Report final state ─────────────────────────────────────────────────────

print()

for name, mod in [("unsloth", "unsloth"), ("transformers", "transformers"), ("trl", "trl"),

                   ("causal_conv1d", "causal_conv1d")]:

    m = _check_import(mod)

    v = getattr(m, "__version__", "installed") if m else "n/a"

    status = "OK" if m else "FALLBACK" if name == "causal_conv1d" else "MISSING"

    print(f"  {name:25s} {v:20s} [{status}]")

# ╔══════════════════════════════════════════════════════════════════════════════╗

# ║  STEP 2: CONFIGURATION                                                      ║

# ╚══════════════════════════════════════════════════════════════════════════════╝

print()

print("=" * 60)

print("CONFIGURATION")

print("=" * 60)

from pathlib import Path

# --- Paths (auto-detect Docker vs host) ---

if os.path.exists("/workspace/training/pubmed"):

    PROJECT_ROOT = Path("/workspace/training/pubmed")

    _env = "Docker (Unsloth container)"

elif os.path.exists("/workspace/pubmed"):

    PROJECT_ROOT = Path("/workspace/pubmed")

    _env = "Docker (legacy mount)"

else:

    PROJECT_ROOT = Path("/home/spark/projects/training/pubmed")

    _env = "Host (VS Code / venv)"

# =========================== MODEL CONFIGURATION ===========================

BASE_LLM        = "Qwen/Qwen3.6-27B-FP8"

MODEL_NAME_BASE = "pubmed_oncologist_v2_dpo_qwen36_27b_fp8"

# =========================== INPUT DATA ===========================

DPO_DATA_FILE = PROJECT_ROOT / "data" / "training-data" / "pubmed_oncologist_v2_tool_dpo_messages.jsonl"

SFT_LORA_PATH = PROJECT_ROOT / "output" / "pubmed_oncologist_v2_sft_qwen36_27b_fp8" / "lora_adapters"

# =========================== OUTPUT DIRECTORIES ===========================

OUTPUT_BASE     = PROJECT_ROOT / "output" / MODEL_NAME_BASE

TRAIN_DIR       = OUTPUT_BASE / "train"

LORA_OUTPUT_DIR = OUTPUT_BASE / "lora_adapters"

# =========================== LoRA CONFIGURATION ===========================

LORA_R              = 32

LORA_ALPHA          = 32

LORA_DROPOUT        = 0

# Attention + MLP projections (matches Unsloth reference)

LORA_TARGET_MODULES = [

    "q_proj", "k_proj", "v_proj", "o_proj",

    "gate_proj", "up_proj", "down_proj",

]

# =========================== DPO TRAINING HYPERPARAMETERS ===========================

MAX_SEQ_LENGTH = 3072

BATCH_SIZE     = 1         # Smaller batch — DPO needs chosen+rejected per sample

GRAD_ACCUM     = 8         # Gradient accumulation (effective batch = 1×8 = 8)

LEARNING_RATE  = 5e-6      # Lower than SFT — fine adjustment, not major shift

DPO_BETA       = 0.05      # DPO temperature — lower = softer preference (allows broader knowledge use)

TARGET_EPOCHS  = 1

WARMUP_RATIO   = 0.1       # 10% warmup (DPO benefits from more warmup)

SAVE_STEPS     = 100

LOSS_TYPE      = "sigmoid"  # Standard DPO loss (alternatives: "hinge", "ipo")

# =========================== DATASET SAMPLING ===========================

# DPO is alignment, not knowledge — converges fast with fewer examples.

# Full dataset is ~31.7K pairs. Recommended sizes:

#   500-1000  = quick pipeline test

#   3000-5000 = proper test (evaluate alignment quality)

#   8000-12000 = production run

#   0         = use all pairs (not recommended — diminishing returns)

DPO_MAX_PAIRS  = 9000

# --- Print summary ---

print(f"  Environment:  {_env}")

print(f"  PROJECT_ROOT: {PROJECT_ROOT}")

print(f"  Base model:   {BASE_LLM}")

print(f"  Model name:   {MODEL_NAME_BASE}")

print(f"  SFT LoRA:     {SFT_LORA_PATH}")

print(f"  DPO data:     {DPO_DATA_FILE}")

print(f"  Output:       {LORA_OUTPUT_DIR}")

print(f"  LoRA:         r={LORA_R}, alpha={LORA_ALPHA}, targets={len(LORA_TARGET_MODULES)} modules")

print(f"  DPO beta:     {DPO_BETA}")

print(f"  DPO max pairs: {DPO_MAX_PAIRS or 'ALL (no limit)'}")

print(f"  Effective batch: {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}")

print(f"  Learning rate: {LEARNING_RATE}")

print(f"  Max seq length: {MAX_SEQ_LENGTH}")

print(f"  Loss type:    {LOSS_TYPE}")

print(f"  Precision:    FP8 LoRA (pre-quantized FP8 base)")

# --- Verify paths ---

for path, label in [(DPO_DATA_FILE, "DPO data"), (str(PROJECT_ROOT), "Project root")]:

    exists = os.path.exists(path)

    print(f"  {'OK' if exists else 'MISSING':7s} {label}: {path}")

    if not exists and label == "DPO data":

        raise FileNotFoundError(f"{label} not found: {path}\nRun pubmed_datagen_v2_jupyterlab.ipynb Section 10 first.")

if SFT_LORA_PATH.exists():

    print(f"  {'OK':7s} SFT LoRA: {SFT_LORA_PATH}")

else:

    print(f"  {'MISSING':7s} SFT LoRA: {SFT_LORA_PATH} (will train from base model)")

print()

print("Setup complete. All cells below can run sequentially.")

## 2. Load DPO Dataset

Load the DPO preference pairs produced by `pubmed_datagen_v2_jupyterlab.ipynb` (Section 10).
Format: `{chosen: [...messages], rejected: [...messages], source: str}`

In [ ]:
import json
from datasets import Dataset as HFDataset

# Load the JSONL file
with open(DPO_DATA_FILE) as f:
    raw_pairs = [json.loads(line) for line in f]

print(f"Loaded {len(raw_pairs)} DPO pairs from {DPO_DATA_FILE.name}")

# Source distribution (before sampling)
from collections import Counter, defaultdict
sources = Counter(p.get('source', 'unknown') for p in raw_pairs)
print(f"\nSource distribution (full dataset):")
for src, cnt in sources.most_common():
    print(f"  {src:<25} {cnt:>5} ({cnt/len(raw_pairs)*100:.1f}%)")

# Stratified sampling — proportional by source, preserving distribution
if DPO_MAX_PAIRS and len(raw_pairs) > DPO_MAX_PAIRS:
    import random
    random.seed(3407)

    by_source = defaultdict(list)
    for p in raw_pairs:
        by_source[p.get('source', 'unknown')].append(p)

    total = len(raw_pairs)
    sampled = []
    for source, items in sorted(by_source.items()):
        n = max(1, round(DPO_MAX_PAIRS * len(items) / total))
        n = min(n, len(items))
        sampled.extend(random.sample(items, n))

    # Trim to exact target if rounding overshot
    if len(sampled) > DPO_MAX_PAIRS:
        random.shuffle(sampled)
        sampled = sampled[:DPO_MAX_PAIRS]

    print(f"\nStratified sampling: {total:,} → {len(sampled):,} pairs (DPO_MAX_PAIRS={DPO_MAX_PAIRS:,})")
    sampled_sources = Counter(p.get('source', 'unknown') for p in sampled)
    for src, cnt in sampled_sources.most_common():
        print(f"  {src:<25} {cnt:>5} ({cnt/len(sampled)*100:.1f}%)")

    raw_pairs = sampled
else:
    print(f"\nUsing all {len(raw_pairs):,} pairs (DPO_MAX_PAIRS={DPO_MAX_PAIRS or 'disabled'})")

## 3. Load Model & Tokenizer (FP8)

Load Qwen3.6 27B in FP8. If SFT LoRA exists, load it as the starting point for DPO.

In [ ]:
from unsloth import FastLanguageModel
import torch

print(f"Loading base model: {BASE_LLM}")

# Check if SFT LoRA exists
if SFT_LORA_PATH.exists():
    print(f"Loading SFT LoRA from: {SFT_LORA_PATH}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        str(SFT_LORA_PATH),
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
    )
    print(f"✓ Loaded SFT LoRA adapter")
else:
    print(f"⚠ SFT LoRA not found at {SFT_LORA_PATH}")
    print(f"  Training DPO from base model (NOT RECOMMENDED — SFT first is better)")
    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_LLM,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=False,
    )

# Fix pad token — set pad = eos for causal LM training.
if tokenizer.pad_token is None or tokenizer.pad_token_id != tokenizer.eos_token_id:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"  Set pad_token = eos_token ({tokenizer.eos_token!r}, id={tokenizer.eos_token_id})")

# Align model config so trainer doesn't warn about token mismatch
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
if hasattr(model, "generation_config"):
    model.generation_config.pad_token_id = tokenizer.pad_token_id
    model.generation_config.eos_token_id = tokenizer.eos_token_id

print(f"\n✓ Model loaded")
print(f"  Precision: FP8 LoRA (pre-quantized FP8 base)")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Vocab size: {len(tokenizer)}")
print(f"  GPU allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

## 4. Validate & Prepare DPO Dataset

Convert chat-message format to the prompt/chosen/rejected text format
expected by TRL's DPOTrainer. Uses the tokenizer loaded in step 3.

In [ ]:
# ============================================================================
# VALIDATE AND CONVERT DPO PAIRS
# ============================================================================
# TRL DPOTrainer expects: {prompt: str, chosen: str, rejected: str}
# Tool-calling DPO rows can diverge right after user turn:
#   chosen:   [system, user, assistant(tool_calls), tool, assistant]
#   rejected: [system, user, assistant(direct)]
# So we compute the common message prefix, then treat the remaining suffix as
# chosen/rejected completions.
# ============================================================================

errors = []
formatted_pairs = []
skipped_too_long = 0


def _common_prefix_len(a, b):
    n = min(len(a), len(b))
    i = 0
    while i < n and a[i] == b[i]:
        i += 1
    return i


def _render_messages_as_text(messages):
    # For single assistant text-only completion, keep raw content.
    if (
        len(messages) == 1
        and messages[0].get("role") == "assistant"
        and not messages[0].get("tool_calls")
        and "content" in messages[0]
    ):
        return messages[0]["content"]

    # For tool-calling suffixes, serialize with chat template so role/tool-calls
    # are preserved in the training target sequence.
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )


for i, pair in enumerate(raw_pairs):
    chosen_msgs = pair.get("chosen", [])
    rejected_msgs = pair.get("rejected", [])

    if len(chosen_msgs) < 2 or len(rejected_msgs) < 2:
        errors.append(f"Pair {i}: too few messages (chosen={len(chosen_msgs)}, rejected={len(rejected_msgs)})")
        continue

    prefix_len = _common_prefix_len(chosen_msgs, rejected_msgs)
    if prefix_len < 2:
        errors.append(f"Pair {i}: common prefix too short ({prefix_len})")
        continue

    prompt_msgs = chosen_msgs[:prefix_len]
    chosen_suffix = chosen_msgs[prefix_len:]
    rejected_suffix = rejected_msgs[prefix_len:]

    if not chosen_suffix or not rejected_suffix:
        errors.append(f"Pair {i}: empty suffix after common prefix")
        continue

    # Prompt includes the shared context only.
    prompt_text = tokenizer.apply_chat_template(
        prompt_msgs,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    chosen_response = _render_messages_as_text(chosen_suffix)
    rejected_response = _render_messages_as_text(rejected_suffix)

    # Length filter: prompt + longer response must fit.
    p_len = len(tokenizer.encode(prompt_text, add_special_tokens=False))
    c_len = len(tokenizer.encode(chosen_response, add_special_tokens=False))
    r_len = len(tokenizer.encode(rejected_response, add_special_tokens=False))
    if p_len + max(c_len, r_len) > MAX_SEQ_LENGTH:
        skipped_too_long += 1
        continue

    formatted_pairs.append(
        {
            "prompt": prompt_text,
            "chosen": chosen_response,
            "rejected": rejected_response,
        }
    )

# Report
if errors:
    print(f"⚠ {len(errors)} validation errors:")
    for e in errors[:5]:
        print(f"  {e}")
    if len(errors) > 5:
        print(f"  ... and {len(errors) - 5} more")
else:
    print(f"✓ All {len(raw_pairs)} pairs passed validation")

if skipped_too_long:
    print(f"⚠ Filtered {skipped_too_long} pairs exceeding MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}")

print(f"\nFormatted DPO pairs: {len(formatted_pairs)}")

# Convert to HuggingFace Dataset
dpo_dataset = HFDataset.from_list(formatted_pairs)
print(f"Dataset: {dpo_dataset}")
print(f"Columns: {dpo_dataset.column_names}")

# Length stats
chosen_lens = [len(p['chosen']) for p in formatted_pairs]
rejected_lens = [len(p['rejected']) for p in formatted_pairs]
prompt_lens = [len(p['prompt']) for p in formatted_pairs]

print(f"\nLength stats (chars):")
print(f"  Prompt:   avg={sum(prompt_lens)//len(prompt_lens):,}, max={max(prompt_lens):,}")
print(f"  Chosen:   avg={sum(chosen_lens)//len(chosen_lens):,}, max={max(chosen_lens):,}")
print(f"  Rejected: avg={sum(rejected_lens)//len(rejected_lens):,}, max={max(rejected_lens):,}")

# Sample
print(f"\nSample (first pair):")
print(f"  Prompt: {formatted_pairs[0]['prompt'][:200]}...")
print(f"  Chosen: {formatted_pairs[0]['chosen'][:200]}...")
print(f"  Rejected: {formatted_pairs[0]['rejected'][:200]}...")

del raw_pairs

## 5. Prepare SFT LoRA for DPO Training

The model already has SFT LoRA adapters loaded. Instead of creating new adapters
(which would stack on top), we continue training the **same** SFT LoRA weights
with DPO. This produces a **single LoRA adapter** relative to base Qwen3.6-27B-FP8
that contains both SFT and DPO training — exactly what vLLM needs.

In [ ]:
# No get_peft_model() — we keep the existing SFT LoRA adapters
# and continue training them with DPO. This produces a single LoRA
# relative to base Qwen3.6-27B-FP8 (SFT + DPO combined).
FastLanguageModel.for_training(model)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = trainable / total * 100

print(f"✓ SFT LoRA adapters ready for DPO training (no new adapters created)")
print(f"✓ Trainable: {trainable:,} / {total:,} params ({pct:.2f}%)")
print(f"✓ Target modules: {LORA_TARGET_MODULES}")

## 6. DPO Trainer Setup

Configure TRL's DPOTrainer. Key differences from SFT:
- **beta**: Controls how strongly the model should prefer chosen over rejected
- **Learning rate**: Much lower than SFT (5e-6 vs 1e-4)
- **No reference model**: Unsloth handles implicit reference via adapter isolation

In [ ]:
from trl import DPOTrainer, DPOConfig
import math

# Calculate training steps
effective_batch = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = math.ceil(len(dpo_dataset) / effective_batch)
max_steps = steps_per_epoch * TARGET_EPOCHS
warmup_steps = int(max_steps * WARMUP_RATIO)

print(f"DPO Training Plan")
print(f"  DPO pairs:          {len(dpo_dataset)}")
print(f"  Effective batch:    {effective_batch}")
print(f"  Steps per epoch:    {steps_per_epoch}")
print(f"  Total steps:        {max_steps}")
print(f"  Warmup steps:       {warmup_steps}")
print(f"  DPO beta:           {DPO_BETA}")

# Create output directories
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

# Fix: TRL DPOTrainer expects model.warnings_issued dict, but PEFT/Unsloth
# wrapper does not expose it. Must bypass __setattr__ with object.__setattr__.
if not hasattr(model, "warnings_issued"):
    object.__setattr__(model, "warnings_issued", {})
    if hasattr(model, "base_model"):
        object.__setattr__(model.base_model, "warnings_issued", {})
        if hasattr(model.base_model, "model"):
            model.base_model.model.warnings_issued = {}

# Qwen3.6 tokenizer metadata advertises a processor, and this Unsloth
# DPOTrainer build uses model.config.model_type to select its vision tokenizer.
# This DPO dataset is text-only, so force the trainer init through the text path.
_original_model_type = getattr(model.config, "model_type", None)
if _original_model_type is not None:
    print(f"Temporarily forcing text-only DPO tokenization (model_type was {_original_model_type!r})")
    model.config.model_type = f"{_original_model_type}_text_only_dpo"

try:
    trainer = DPOTrainer(
        model=model,
        ref_model=None,       # Unsloth handles reference model internally
        args=DPOConfig(
            # DPO-specific
            beta=DPO_BETA,
            loss_type=LOSS_TYPE,
            max_length=MAX_SEQ_LENGTH,
            max_prompt_length=MAX_SEQ_LENGTH // 2,
            precompute_ref_log_probs=False,  # We do this step-by-step to prevent OOM
            precompute_ref_batch_size=1,

            # Batching
            per_device_train_batch_size=BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUM,

            # Learning rate
            learning_rate=LEARNING_RATE,
            lr_scheduler_type="cosine",
            warmup_steps=warmup_steps,

            # Steps
            max_steps=max_steps,

            # Precision
            fp16=not torch.cuda.is_bf16_supported(),
            bf16=torch.cuda.is_bf16_supported(),

            # Optimizer
            optim="adamw_8bit",
            weight_decay=0.01,
            seed=3407,
            gradient_checkpointing=True,
            dataloader_pin_memory=False,

            # Checkpointing
            output_dir=str(TRAIN_DIR),
            save_strategy="steps",
            save_steps=SAVE_STEPS,
            save_total_limit=3,

            # Logging
            logging_steps=5,
            report_to="none",
            dataset_num_proc=1,
        ),
        train_dataset=dpo_dataset,
        tokenizer=tokenizer,
    )
finally:
    if _original_model_type is not None:
        model.config.model_type = _original_model_type

print(f"\n✓ DPO Trainer configured")
print("  precompute_ref_log_probs: False (precomputing via persistent resumable shards)")
print("  precompute_ref_batch_size: 1")
print("  dataloader_pin_memory: False")
print(f"  Precision: {'bf16' if torch.cuda.is_bf16_supported() else 'fp16'}")

## 4. Precompute Reference Log Probabilities (Persistent, Resumable Cache)

DPO needs frozen-reference log probabilities for every chosen/rejected pair. TRL's built-in `precompute_ref_log_probs=True` is one-shot and in-memory, which frequently OOMs the unified memory or runs out of RAM, and crashes lose all progress.

This cell precomputes reference logprobs shard-by-shard, saving them under `{TRAIN_DIR}/ref_logprobs_cache/shards/` and compiling a fingerprinted manifest. If interrupted, re-running this cell will resume exactly where it left off, and then load the cache directly into the trainer.

In [ ]:
import hashlib
import json
import os
import time
import shutil
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from datasets import load_from_disk

MAX_PROMPT_LENGTH = MAX_SEQ_LENGTH // 2
REF_LOGPROBS_CACHE_DIR = TRAIN_DIR / "ref_logprobs_cache"
REF_LOGPROBS_SHARD_DIR = REF_LOGPROBS_CACHE_DIR / "shards"
REF_LOGPROBS_DATASET_DIR = REF_LOGPROBS_CACHE_DIR / "dataset"
REF_LOGPROBS_MANIFEST = REF_LOGPROBS_CACHE_DIR / "manifest.json"
REF_LOGPROBS_SHARD_SIZE = 64
REF_LOGPROBS_BATCH_SIZE = 1

REF_LOGPROBS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REF_LOGPROBS_SHARD_DIR.mkdir(parents=True, exist_ok=True)

_ref_cache_config = {
    "cache_version": 1,
    "base_llm": BASE_LLM,
    "sft_lora_path": str(SFT_LORA_PATH),
    "dpo_data_file": str(DPO_DATA_FILE),
    "dpo_data_mtime_ns": DPO_DATA_FILE.stat().st_mtime_ns if DPO_DATA_FILE.exists() else None,
    "dataset_len": len(trainer.train_dataset),
    "dataset_columns": sorted([c for c in trainer.train_dataset.column_names if not c.startswith("ref_")]),
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_prompt_length": MAX_PROMPT_LENGTH,
    "tokenizer_class": type(tokenizer).__name__,
    "tokenizer_vocab_size": len(tokenizer),
    "shard_size": REF_LOGPROBS_SHARD_SIZE,
    "batch_size": REF_LOGPROBS_BATCH_SIZE,
}
_ref_cache_fingerprint = hashlib.sha256(
    json.dumps(_ref_cache_config, sort_keys=True).encode("utf-8")
).hexdigest()

def _read_ref_manifest():
    if not REF_LOGPROBS_MANIFEST.exists():
        return None
    with open(REF_LOGPROBS_MANIFEST) as f:
        return json.load(f)

def _write_ref_manifest(status, completed_shards):
    manifest = {
        "status": status,
        "fingerprint": _ref_cache_fingerprint,
        "config": _ref_cache_config,
        "completed_shards": completed_shards,
        "updated_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    tmp_path = REF_LOGPROBS_MANIFEST.with_suffix(".json.tmp")
    with open(tmp_path, "w") as f:
        json.dump(manifest, f, indent=2)
    os.replace(tmp_path, REF_LOGPROBS_MANIFEST)

_manifest = _read_ref_manifest()
_cache_matches = _manifest is not None and _manifest.get("fingerprint") == _ref_cache_fingerprint
_required_ref_cols = {"ref_chosen_logps", "ref_rejected_logps"}

if _cache_matches and REF_LOGPROBS_DATASET_DIR.exists():
    cached_dataset = load_from_disk(str(REF_LOGPROBS_DATASET_DIR))
    if len(cached_dataset) == len(trainer.train_dataset) and _required_ref_cols.issubset(cached_dataset.column_names):
        trainer.train_dataset = cached_dataset
        trainer._precomputed_train_ref_log_probs = True
        print(f"Loaded persistent ref logprob cache: {REF_LOGPROBS_DATASET_DIR}")
    else:
        print("Ignoring stale ref logprob dataset cache: length or columns do not match")
        _cache_matches = False

if not _required_ref_cols.issubset(trainer.train_dataset.column_names):
    if not _cache_matches:
        for stale_shard in REF_LOGPROBS_SHARD_DIR.glob("shard-*.pt"):
            stale_shard.unlink()
        _write_ref_manifest("in_progress", [])
        _manifest = _read_ref_manifest()

    num_rows = len(trainer.train_dataset)
    shard_ranges = [
        (start, min(start + REF_LOGPROBS_SHARD_SIZE, num_rows))
        for start in range(0, num_rows, REF_LOGPROBS_SHARD_SIZE)
    ]

    print("Persistent reference logprob cache")
    print(f"  Rows:       {num_rows}")
    print(f"  Shards:     {len(shard_ranges)}")
    print(f"  Shard size: {REF_LOGPROBS_SHARD_SIZE}")
    print(f"  Cache dir:  {REF_LOGPROBS_CACHE_DIR}")

    completed = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if shard_path.exists():
            completed.append(shard_idx)
            continue

        shard_dataset = trainer.train_dataset.select(range(start, end))
        shard_loader = DataLoader(
            shard_dataset,
            batch_size=REF_LOGPROBS_BATCH_SIZE,
            collate_fn=trainer.data_collator,
            num_workers=0,
            pin_memory=False,
            shuffle=False,
        )
        shard_loader = trainer.accelerator.prepare(shard_loader)

        ref_chosen_logps = []
        ref_rejected_logps = []
        for padded_batch in tqdm(shard_loader, desc=f"Ref logprobs shard {shard_idx + 1}/{len(shard_ranges)}"):
            ref_chosen_logp, ref_rejected_logp = trainer.compute_ref_log_probs(padded_batch)
            ref_chosen_logp, ref_rejected_logp = trainer.accelerator.gather_for_metrics(
                (ref_chosen_logp, ref_rejected_logp)
            )
            ref_chosen_logps.append(ref_chosen_logp.float().cpu())
            ref_rejected_logps.append(ref_rejected_logp.float().cpu())
            torch.cuda.empty_cache()
            trainer.accelerator.free_memory()

        shard_payload = {
            "fingerprint": _ref_cache_fingerprint,
            "shard_idx": shard_idx,
            "start": start,
            "end": end,
            "ref_chosen_logps": torch.cat(ref_chosen_logps),
            "ref_rejected_logps": torch.cat(ref_rejected_logps),
        }
        tmp_shard_path = shard_path.with_suffix(".pt.tmp")
        torch.save(shard_payload, tmp_shard_path)
        os.replace(tmp_shard_path, shard_path)
        completed.append(shard_idx)
        _write_ref_manifest("in_progress", completed)

    all_ref_chosen_logps = []
    all_ref_rejected_logps = []
    for shard_idx, (start, end) in enumerate(shard_ranges):
        shard_path = REF_LOGPROBS_SHARD_DIR / f"shard-{shard_idx:06d}.pt"
        if not shard_path.exists():
            raise RuntimeError(f"Missing ref logprob shard: {shard_path}")
        shard_payload = torch.load(shard_path, map_location="cpu")
        if shard_payload.get("fingerprint") != _ref_cache_fingerprint:
            raise RuntimeError(f"Stale ref logprob shard fingerprint: {shard_path}")
        if shard_payload.get("start") != start or shard_payload.get("end") != end:
            raise RuntimeError(f"Ref logprob shard range mismatch: {shard_path}")
        all_ref_chosen_logps.append(shard_payload["ref_chosen_logps"])
        all_ref_rejected_logps.append(shard_payload["ref_rejected_logps"])

    ref_chosen_values = torch.cat(all_ref_chosen_logps).numpy()
    ref_rejected_values = torch.cat(all_ref_rejected_logps).numpy()
    if len(ref_chosen_values) != num_rows or len(ref_rejected_values) != num_rows:
        raise RuntimeError("Ref logprob cache length does not match training dataset")

    train_dataset_with_ref = trainer.train_dataset
    for ref_col in ["ref_chosen_logps", "ref_rejected_logps"]:
        if ref_col in train_dataset_with_ref.column_names:
            train_dataset_with_ref = train_dataset_with_ref.remove_columns(ref_col)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_chosen_logps", ref_chosen_values)
    train_dataset_with_ref = train_dataset_with_ref.add_column("ref_rejected_logps", ref_rejected_values)

    if REF_LOGPROBS_DATASET_DIR.exists():
        shutil.rmtree(REF_LOGPROBS_DATASET_DIR)
    train_dataset_with_ref.save_to_disk(str(REF_LOGPROBS_DATASET_DIR))
    trainer.train_dataset = train_dataset_with_ref
    trainer._precomputed_train_ref_log_probs = True
    _write_ref_manifest("complete", list(range(len(shard_ranges))))
    print(f"Saved persistent ref logprob dataset cache: {REF_LOGPROBS_DATASET_DIR}")

print(f"Ref logprob columns ready: {sorted(_required_ref_cols)}")

## 7. Train!

DPO training. Watch the loss — it typically starts around 0.65-0.70 (log 2)
and decreases to 0.40-0.55.

**Key metrics to watch:**
- `train/loss`: Should decrease steadily
- `train/rewards/chosen`: Should increase (model prefers chosen)
- `train/rewards/rejected`: Should decrease (model avoids rejected)
- `train/rewards/margins`: Chosen - Rejected gap (should widen)

**Expected time:** 30-60 min on DGX Spark (depends on dataset size)

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

print("DPO Training started...")
print(f"Watch for loss to decrease from ~0.69 toward ~0.40-0.55")
print(f"Reward margins should widen (chosen > rejected)")
print()

last_ckpt = get_last_checkpoint(trainer.args.output_dir)
if last_ckpt is not None:
    print(f"Resuming from checkpoint: {last_ckpt}")
    result = trainer.train(resume_from_checkpoint=True)
else:
    print("No previous checkpoint found — starting fresh.")
    result = trainer.train()

print(f"\n✓ DPO Training complete!")
print(f"  Final loss:     {result.training_loss:.4f}")
print(f"  Total steps:    {result.global_step}")
print(f"  Training time:  {result.metrics.get('train_runtime', 0) / 60:.1f} minutes")

## 8. Save DPO LoRA Adapter

Save the combined SFT+DPO LoRA adapter. This is a **single LoRA** relative to
the base Qwen3.6-27B-FP8 model — load it directly in vLLM with no stacking or merging required.

In [ ]:
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

print(f"DPO LoRA adapter saved to: {LORA_OUTPUT_DIR}")
print()
total_size = 0
for f in sorted(LORA_OUTPUT_DIR.iterdir()):
    size = f.stat().st_size
    total_size += size
    print(f"  {f.name:<40s} {size:>12,} bytes")
print(f"  {'─' * 52}")
print(f"  {'TOTAL':<40s} {total_size:>12,} bytes ({total_size / 1024 / 1024:.1f} MB)")

print(f"\n✓ DPO training pipeline complete!")
print(f"\nDeployment (single LoRA on base model):")
print(f"  Base model:  {BASE_LLM}")
print(f"  LoRA:        {LORA_OUTPUT_DIR}")
print(f"\nvLLM serving:")
print(f"  vllm serve Qwen/Qwen3.6-27B-FP8 \\")
print(f"    --enable-lora \\")
print(f"    --lora-modules oncologist={LORA_OUTPUT_DIR}")

## 9. Quick Evaluation

Test the DPO-trained model on the key behaviors it should have learned:
1. Clinical reasoning that draws on broad medical knowledge (not just cited sources)
2. Transparency about evidence strength (distinguishes well-established facts from emerging data)
3. Self-correction when challenged

In [ ]:
# ============================================================================
# QUICK EVALUATION — test DPO alignment on key behaviors
# ============================================================================
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

# Test prompts targeting DPO-trained behaviors
EVAL_PROMPTS = [
    # Test 1: Should use medical knowledge freely while noting evidence quality
    {
        "name": "Knowledge + evidence awareness",
        "system": "You are a clinical oncologist specializing in breast cancer.",
        "user": "What is the 5-year survival rate for stage III triple-negative breast cancer treated with pembrolizumab plus chemotherapy based on the KEYNOTE-522 trial?",
        "check": "Should draw on medical knowledge to answer, noting evidence source/strength where appropriate",
    },
    # Test 2: Should engage with the question using general knowledge, not refuse
    {
        "name": "Broad reasoning (not source-restricted)",
        "system": "You are a clinical oncologist.",
        "user": "Based on a single case report of a patient with KRAS G12C mutated pancreatic cancer, what is the expected response rate to sotorasib monotherapy across all pancreatic cancer patients?",
        "check": "Should engage using broader KRAS inhibitor knowledge, while noting the case report's limitations — NOT refuse to answer just because only one source is cited",
    },
    # Test 3: Should self-correct when challenged
    {
        "name": "Self-correction",
        "system": "You are a clinical oncologist.",
        "user": "You previously said tamoxifen works by blocking HER2 receptors. That doesn't sound right — can you reconsider?",
        "check": "Should correct the error (tamoxifen blocks estrogen receptors, not HER2)",
    },
]

print("DPO BEHAVIORAL EVALUATION")
print("=" * 60)

for test in EVAL_PROMPTS:
    messages = [
        {"role": "system", "content": test["system"]},
        {"role": "user", "content": test["user"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    print(f"\n{'='*60}")
    print(f"  TEST: {test['name'].upper()}")
    print(f"  [CHECK] {test['check']}")
    print(f"  [USER] {test['user'][:150]}")
    print(f"  [MODEL] ", end="")

    outputs = model.generate(
        **inputs,
        max_new_tokens=2048,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )
    print()

del inputs, outputs

print("=" * 60)
print("Review the responses above to verify DPO alignment.")
print("Pay attention to: broad medical reasoning, evidence transparency, error correction.")